# RAG

With SageMaker.
Source: https://github.com/ExamProCo/AWS-Examples/blob/main/bedrock/rag/rag-with-haiku.ipynb

## Setup

In [ ]:
%pip install -q -U langchain-aws

In [ ]:
!pip show boto3 botocore

# Refresh all variables, aliases and directory history. from store
%store -r

# restart kernel

from IPython.core.display import HTML
HTML("<script>Jupyter.notebook.kernel.restart()</script>")

In [ ]:
import boto3
import json

Set both client and agent.

In [ ]:
client = boto3.client("bedrock-runtime")
agent_client = boto3.client("bedrock-agent-runtime")

## Retrieve API

Call the `Retreive API` (using agent) provided by Knowledge Bases to convert user queries into embeddings, searches the knowledge base, and returns the relevant results.

This gives you more control to build custom workﬂows on top of the semantic search results. 

**Output**
The output of the Retrieve API includes:
    - the `retrieved text chunks`, 
    - the `location type` and 
    - `URI` of the source data,
    - relevance `scores` of the retrievals. 

By default, auto-selects the right strategy to give most relevant results. To override the default optio, set the value to `HYBRID`/`SEMANTIC` in `retrievalConfiguration` > `overrideSearchType`

In [ ]:
def retrieve(query, kbId, numberOfResults=5):
    return agent_client.retrieve(
        retrievalQuery= {'text': query},
        knowledgeBaseId=kbId,
        retrievalConfiguration= {
            'vectorSearchConfiguration': {
                'numberOfResults': numberOfResults,
                'overrideSearchType': "HYBRID", # optional
            }
        }
    )

Call the retreive API, and pass knowledge base id, number of results and query as paramters.

`score`: You can view the associated score of each of the text chunk that was returned which depicts its correlation to the query in terms of how closely it matches it.

In [ ]:
query = "What is Amazon doing in the field of generative AI?"
kb_id = "RD3KHHLG96"

In [ ]:
response = retrieve(query, kb_id, 5)
retrievalResults = response['retrievalResults']

In [ ]:
import pprint
pp = pprint.PrettyPrinter(indent=2)
pp.pprint(retrievalResults)

### Extract the text chunks

From the retrieveAPI response, fetch the context from the retrieval results.

In [ ]:
def get_contexts(retrievalResults):
    contexts = []
    for retrievedResult in retrievalResults: 
        contexts.append(retrievedResult['content']['text'])
    return contexts

In [ ]:
contexts = get_contexts(retrievalResults)
pp.pprint(contexts)

## Data (Prompt)

 Model acts as a financial advisor AI system that will provide answers to questions by using fact based and statistical information when possible. 
 
 We will provide the Retrieve API responses from above as a part of the {contexts} in the prompt for the model to refer to, along with the user query.

 *Note: Change the prompt based on the KB you have.*

In [ ]:
prompt_data = f"""
Human: You are a financial advisor AI system, and provides answers to questions by using fact based and statistical information when possible. 
Use the following pieces of information to provide a concise answer to the question enclosed in <question> tags. 
If you don't know the answer, just say that you don't know, don't try to make up an answer.
<context>
{contexts}
</context>

<question>
{query}
</question>

The response should be specific and use statistics or numbers when possible.

Assistant:"""

## Call API

In [ ]:
body = json.dumps({
    "prompt": prompt_data,
    "max_tokens":512,
    "temperature":0.5,
    "top_k":50,
    "top_p":0.9
})

In [ ]:
resp = client.invoke_model(
    modelId='mistral.mistral-7b-instruct-v0:2',
    contentType='application/json',
    accept='application/json',
    body=body
)

In [ ]:
body = json.loads(resp.get("body").read())
answer = body['outputs'][0]['text']
print(answer)